# Estimator-based VQE energy scan

Evaluate a compact two-qubit chemistry-style Hamiltonian over a variational ansatz parameter scan.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

VQE repeatedly evaluates a parameterized ansatz against a Hamiltonian. Simulator overhead is paid once per energy evaluation.

In [2]:
hamiltonian = SparsePauliOp.from_list([
    ("II", -1.05), ("ZI", 0.39), ("IZ", -0.39), ("ZZ", -0.01), ("XX", 0.18)
])
angles = np.linspace(-np.pi, np.pi, 25)
circuits = []
for angle in angles:
    circuit = QuantumCircuit(2)
    circuit.ry(float(angle), 0)
    circuit.cx(0, 1)
    circuit.ry(float(-0.37 * angle), 1)
    circuits.append(circuit)

def reference_energies():
    estimator = StatevectorEstimator()
    return np.asarray([estimator.run([(c, hamiltonian)]).result()[0].data.evs.item() for c in circuits])

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(reference_energies)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="cpu")
compiled = [transpile(c, backend, optimization_level=1) for c in circuits]
estimator = MettleQEstimatorV2(backend=backend)

def mettleq_energies():
    return np.asarray([estimator.run([(c, hamiltonian)]).result()[0].data.evs.item() for c in compiled])

candidate, mettleq_ms, _ = benchmark(mettleq_energies)
error = max_abs_error(reference, candidate)
minima_match = int(np.argmin(reference)) == int(np.argmin(candidate))
method, device = qiskit_selection(estimator)

## 4. Check correctness before discussing speed

The complete energy trace is compared, not merely the final minimum.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/10_vqe.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="VQE energy trace atol=3e-6",
    passed=error <= 3e-6 and minima_match,
    exact_match=minima_match,
    selected_method=method,
    selected_device=device,
    metrics={"max_energy_error": error, "reference_minimum": float(reference.min()), "mettleq_minimum": float(candidate.min()), "minimum_index": int(np.argmin(candidate))},
)


Comparison summary
------------------
Correctness contract: PASS — VQE energy trace atol=3e-6
SDK reference median: 16.987 ms
MettleQ median:       120.086 ms
Timing interpretation: the SDK reference was 7.069x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: yes

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "VQE energy trace atol=3e-6", "exact_match": true, "framework": "qiskit", "machine": "arm64", "metrics": {"max_energy_error": 1.3451273783715578e-07, "mettleq_minimum": -1.2243290054798128, "minimum_index": 7, "reference_minimum": -1.2243289931098607}, "mettleq_median_ms": 120.08591601625085, "notebook": "qiskit/10_vqe.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 16.986750008072704, "reference_over_mettleq": 0.14145497300261206, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


## What should you conclude?

Real advantage requires wider ansatzes or batched evaluations; a two-qubit VQE is dominated by Python and adapter overhead.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.